In [ ]:
## Wrapper function
source('shearwater_flt_2019_WT_mt22.R')

In [ ]:
tumour_dict = list(
    'AT3'=c('PD60967', 'PD61833a'),
    'AT6'=c('PD63561', 'PD61836a'),
    'AT10'=c('PD60966', 'PD61840a'),
    'AT14'=c('PD63564', 'PD64540b'),
    'AT15'=c('PD63565', 'PD64541b')
)

In [ ]:
# select sample info
N=5
donor_id = names(tumour_dict)[N]
tumour_id = tumour_dict[[N]][1]
normal_id = tumour_dict[[N]][2]
print(donor_id)

[1] "AT15"


In [ ]:
# Your sample of interest
samples_fp = sprintf("../data/LCM_analysis/metadata/cgpVAF_runs/%s.txt", donor_id)
samples = read.table(samples_fp, header=F) 
samples_PDID = samples$V1

case_samples_PDID = samples_PDID[grepl(tumour_id, samples_PDID)]
normal_samples_PDID = samples_PDID[!grepl(tumour_id, samples_PDID)]

# Outdirs
outDir = sprintf('../data/LCM_analysis/shearwater_output/%s/', tumour_id)
dir.create(outDir, showWarnings = FALSE)
alleleCount_outDir = paste0(outDir, 'allele_count/')
dir.create(alleleCount_outDir, showWarnings = FALSE)

# Prep for allele count
cgp_out_fp = '../data/LCM_analysis/cgpVAF/'
cgp_out_files = list.files(cgp_out_fp)
cgpVAF_output_fp = paste0(cgp_out_fp, 
                          cgp_out[grepl(normal_id, cgp_out_files) & grepl('tsv', cgp_out_files)])

In [ ]:
#extract_alleleCount(outDir,cgpVAF_output_fp,alleleCount_outDir,
#                    samples_PDID = c(case_samples_PDID,
#                                     normal_samples_PDID))

In [ ]:
# Read all lines from the file beginning with #
lines = readLines(cgpVAF_output_fp)
lines_to_skip = sum(grepl("^#", lines))-1

cgpVAF_output = read.table(cgpVAF_output_fp,
header = T, sep = '\t', stringsAsFactors = F, comment.char = "", skip = lines_to_skip)

colnames(cgpVAF_output)[which(colnames(cgpVAF_output)=="VariantID")]="ID"
row.names(cgpVAF_output)=paste0(sub('.*chr', '', cgpVAF_output$Chrom), "_", cgpVAF_output$Pos)

In [ ]:
allele_count = cgpVAF_output[,c('Chrom','Pos')]
  
# Extract base count for each sample
for (i in seq_along(samples_PDID)) {
    sample = samples_PDID[i]
    print(sample)
    # Check if it is present in cgpVAF output
    if(sum(grepl(paste0(sample,"_"),colnames(cgpVAF_output))) == 0){
      warning(sprintf('No cgpVAF output for sample %s',sample))
      next()
    }else{
      mut_temp=cgpVAF_output[,grep(paste0(sample,"_"), colnames(cgpVAF_output))]  
      #reset the allele counts to NA before starting a new sample. Not strictly necessary but doing it for my sanity/piece of mind
      allele_count$Count_A = mut_temp[,grepl('FAZ',colnames(mut_temp))] + mut_temp[,grepl('RAZ',colnames(mut_temp))]
      allele_count$Count_C = mut_temp[,grepl('FCZ',colnames(mut_temp))] + mut_temp[,grepl('RCZ',colnames(mut_temp))]
      allele_count$Count_G = mut_temp[,grepl('FGZ',colnames(mut_temp))] + mut_temp[,grepl('RGZ',colnames(mut_temp))]
      allele_count$Count_T = mut_temp[,grepl('FTZ',colnames(mut_temp))] + mut_temp[,grepl('RTZ',colnames(mut_temp))]
    }
    
    write.table(allele_count, file = file.path(alleleCount_outDir,paste0(sample, "_alleleCounts.txt")),
                sep = "\t", row.names = F, quote = F, col.names = T)
  }
  
print('STEP2: Completed')

[1] "PD63565d_lo0016"
[1] "PD63565e_lo0015"
[1] "PD63565a_lo0013"
[1] "PD63565a_lo0015"
[1] "PD55504c"
[1] "PD55505c"
[1] "PD55506c"
[1] "PD55508c"
[1] "PD55509c"
[1] "PD55510c"
[1] "PD55511c"
[1] "PD55512c"
[1] "PD55513c"
[1] "PD55515c"
[1] "PD55516c"
[1] "PD55517c"
[1] "PD55518c"
[1] "PD55519c"
[1] "PD55520c"
[1] "PD55521c"
[1] "PD55522c"
[1] "PD55524c"
[1] "PD55526c"
[1] "PD55527c"
[1] "PD55528c"
[1] "PD55529c"
[1] "PD55530c"
[1] "PD55531c"
[1] "PD55533c"
[1] "PD55534c"
[1] "PD55535c"
[1] "PD55536c"
[1] "PD55538c"
[1] "STEP2: Completed"


In [ ]:
variants = cgpVAF_output[,c('Chrom', 'Pos', 'Ref', 'Alt')]
colnames(variants) = c('Chr', 'Pos', 'Ref', 'Alt')
coords=paste(variants$Chr,variants$Pos,sep="_")
rownames(variants) = coords

In [ ]:

variants_SW = runShearWaterLike(case_samples_PDID=case_samples_PDID,
                                normal_samples_PDID = normal_samples_PDID,
                                outDir=outDir,patient=tumour_id,
                                variants = variants,alleleCount_outDir = alleleCount_outDir)

[1] "STEP3: Run Tims shearwater-like filter to remove false mutations"
[1] 4
[1] "PD63565d_lo0016"
[1] "PD63565e_lo0015"
[1] "PD63565a_lo0013"
[1] "PD63565a_lo0015"
[1] "PD55504c"
[1] "PD55505c"
[1] "PD55506c"
[1] "PD55508c"
[1] "PD55509c"
[1] "PD55510c"
[1] "PD55511c"
[1] "PD55512c"
[1] "PD55513c"
[1] "PD55515c"
[1] "PD55516c"
[1] "PD55517c"
[1] "PD55518c"
[1] "PD55519c"
[1] "PD55520c"
[1] "PD55521c"
[1] "PD55522c"
[1] "PD55524c"
[1] "PD55526c"
[1] "PD55527c"
[1] "PD55528c"
[1] "PD55529c"
[1] "PD55530c"
[1] "PD55531c"
[1] "PD55533c"
[1] "PD55534c"
[1] "PD55535c"
[1] "PD55536c"
[1] "PD55538c"
[1] "STEP3: Shearwater-like filter completed!"


In [ ]:
print('FDR = 0.001')
table(variants_SW[,5] < 0.001)
print('FDR = 0.005')
table(variants_SW[,5] < 0.005)

[1] "FDR = 0.001"



FALSE  TRUE 
 2848  5546 

[1] "FDR = 0.005"



FALSE  TRUE 
 2733  5661 